In [ ]:
# SEGMENTATION NON SUPERVISÉE PAR SEUILLAGE ADAPTATIF (MEAN)

import numpy as np
import tifffile
import matplotlib.pyplot as plt
from skimage import filters, exposure
from skimage.filters import threshold_local
from scipy import ndimage as ndi
import os
import random


In [ ]:
#tiff_path = "./data_yukiko/images/default/triax.0.eps=-0.0025.spheres_image_gaussian=08_noise=03.tif"
tiff_path = "./data_yukiko/images/basic/triax.0.eps=-0.0025.spheres_image_gaussian=00_noise=00.tif"

with tifffile.TiffFile(tiff_path) as tif:
    image = tif.asarray()
    print(f"Dimensions : {image.shape}")
    print(f"Type : {image.dtype}")


In [ ]:
def extract_subvolume(image, region_size=100, offset_range=50):
    d, h, w = image.shape
    cz, cy, cx = d//2, h//2, w//2
    oz, oy, ox = [random.randint(-offset_range, offset_range) for _ in range(3)]

    sz = np.clip(cz + oz - region_size//2, 0, d - region_size)
    sy = np.clip(cy + oy - region_size//2, 0, h - region_size)
    sx = np.clip(cx + ox - region_size//2, 0, w - region_size)

    return image[sz:sz+region_size, sy:sy+region_size, sx:sx+region_size]

subvolume = extract_subvolume(image, region_size=100)
print("Shape sous-volume :", subvolume.shape)


In [ ]:
from skimage.filters import laplace, gaussian

def preprocess_log(volume, sigma=1.0):
    vol = volume.astype(np.float32)
    vol -= vol.min()
    vol /= vol.max()
    
    smoothed = gaussian(vol, sigma=sigma, preserve_range=True)
    log = laplace(smoothed)
    
    # Inversion pour faire ressortir les grains clairs
    return -log

preprocessed = preprocess_log(subvolume, sigma=1.2)


In [ ]:
z_slice = preprocessed.shape[0] // 2
plt.imshow(preprocessed[z_slice], cmap='gray')
plt.title(f'Slice Z={z_slice} - Prétraitement amélioré')
plt.axis('off')
plt.show()


In [ ]:
from skimage.filters import threshold_local

def adaptive_threshold_3d(volume, block_size=35, offset=0.01):
    binary = np.zeros_like(volume, dtype=bool)
    
    for z in range(volume.shape[0]):
        slice_ = volume[z]
        thresh = threshold_local(slice_, block_size=block_size, offset=offset)
        binary[z] = slice_ > thresh

    return binary.astype(np.uint8)

binary_mask = adaptive_threshold_3d(preprocessed, block_size=55, offset=0.01)


In [ ]:
z = binary_mask.shape[0] // 2
plt.imshow(binary_mask[z], cmap='gray')
plt.title(f"Z={z} — Seuillage adaptatif binaire")
plt.axis('off')
plt.show()


In [ ]:
from skimage.morphology import remove_small_objects
from skimage.measure import label

def clean_and_label(binary_mask, min_size=200):
    # Inverser : grains = 1, fond = 0
    mask = binary_mask == 0

    # Supprimer petits objets parasites
    mask_clean = remove_small_objects(mask, min_size=min_size)

    # Étiqueter chaque grain 3D
    labels = label(mask_clean)

    return labels


labels = clean_and_label(binary_mask, min_size=200)


In [ ]:

def plot_segmentation(labels):
    fig, axes = plt.subplots(2, 4, figsize=(15, 8))
    fig.suptitle('Segmentation - Clustered Slices', fontsize=16)

    slice_indices = [20, 40, 60, 80]

    for i, slice_idx in enumerate(slice_indices):
        col = i
        row = 0

        im = axes[row, col].imshow(labels[slice_idx], cmap='nipy_spectral', interpolation='nearest')
        axes[row, col].set_title(f'Z-slice {slice_idx} (filtered)')
        axes[row, col].set_xlabel('X')
        axes[row, col].set_ylabel('Y')
        plt.colorbar(im, ax=axes[row, col], shrink=0.5)

        row = 1
        im = axes[row, col].imshow(preprocessed[slice_idx], cmap='gray')
        axes[row, col].set_title(f'Z-slice {slice_idx}')
        axes[row, col].set_xlabel('X')
        axes[row, col].set_ylabel('Y')
        plt.colorbar(im, ax=axes[row, col], shrink=0.5)

    plt.tight_layout()
    plt.show()


plot_segmentation(labels)  # réutilise ta fonction de visualisation existante
